In [4]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel

SectionName = Literal[
    "overview",
    "seed",
    "land_preparation",
    "intercultural",
    "irrigation",
    "harvest",
    "fertilizer",
    "climate",
    "variety",
    "pesticide",
    "herbicide",
]

PROMPT = """
Classify the user's agricultural query into one or more sections.

Available sections:

- overview: general information about a crop
- seed: seed rate, seed quality, seed treatment, sowing
- land_preparation: soil preparation, ploughing, bed preparation
- intercultural: weeding, thinning, pruning, crop care
- irrigation: watering and irrigation
- harvest: harvesting time and harvesting method
- fertilizer: fertilizer, manure and nutrient application
- climate: temperature, rainfall, season, soil and weather requirements
- variety: crop varieties, variety comparison and variety selection
- pesticide: pests, diseases, insecticides and pesticides
- herbicide: weeds and herbicide application

Return every relevant section.
Only return values from the available section list.
"""

class SectionResult(BaseModel):
    sections: list[SectionName]

llm = ChatOllama(model="gemma3:4b", temperature=0)

structured_llm = llm.with_structured_output(SectionResult)

prompt = ChatPromptTemplate.from_messages([
    ("system", PROMPT),
    ("human", "{query}")
])

chain = prompt | structured_llm
query = input("Enter your query: ")
result = chain.invoke({"query": query})

In [7]:
from __future__ import annotations

import json
import time
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field

SectionName = Literal[
    "overview",
    "seed",
    "land_preparation",
    "intercultural",
    "irrigation",
    "harvest",
    "fertilizer",
    "climate",
    "variety",
    "pesticide",
    "herbicide",
]


class SectionResult(BaseModel):
    sections: list[SectionName] = Field(
        default_factory=list,
        description=(
            "Every relevant agricultural section. "
            "Return an empty list when no section is detected."
        ),
    )

PROMPT = """
Classify the user's agricultural query into one or more relevant sections.

Available sections:

- overview: general crop information or a broad cultivation overview
- seed: seed selection, seed rate, treatment, nursery, sowing, planting or transplanting
- land_preparation: tillage, ploughing, leveling, beds, pits or field preparation
- intercultural: crop-care operations after planting, excluding irrigation, fertilizer and chemical protection
- irrigation: watering, irrigation scheduling, drainage or water management
- harvest: maturity, harvesting, yield, post-harvest handling or storage
- fertilizer: fertilizers, manure, nutrients, deficiencies, doses or application schedules
- climate: soil suitability, season, temperature, rainfall, humidity, sunlight or other growing conditions
- variety: variety identification, selection, characteristics, suitability or comparison
- pesticide: pests, diseases and their prevention, diagnosis or control
- herbicide: chemical control of weeds

Rules:

1. Classify by meaning, not by exact words or language.
2. Return every section directly requested by the query.
3. Do not include sections that are only indirectly related.
4. Distinguish sections according to the purpose of the requested information.
5. A query may belong to multiple sections.
6. For a broad end-to-end cultivation question, return all major sections needed to answer it.
7. For general crop information without a specific topic, return only overview.
8. Return no duplicate sections.
9. If no section can be confidently identified, return an empty list.

Output only the structured result.
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", PROMPT),
        ("human", "{query}"),
    ]
)


llm = ChatOllama(
    model="gemma3:4b",
    temperature=0,
)


classifier = prompt | llm.with_structured_output(SectionResult)


TEST_CASES: list[dict] = [
    # Seed: 1-10
    {
        "query": "What is the seed rate for aman rice?",
        "expected": ["seed"],
    },
    {
        "query": "How much seed is required for one acre of wheat?",
        "expected": ["seed"],
    },
    {
        "query": "How should I treat maize seeds before sowing?",
        "expected": ["seed"],
    },
    {
        "query": "What is the correct sowing distance for onion?",
        "expected": ["seed"],
    },
    {
        "query": "When should I sow mustard seeds?",
        "expected": ["seed"],
    },
    {
        "query": "বোরো ধানের বীজের হার কত?",
        "expected": ["seed"],
    },
    {
        "query": "বীজ শোধন কীভাবে করব?",
        "expected": ["seed"],
    },
    {
        "query": "ধানের চারা কত দিনের হলে রোপণ করব?",
        "expected": ["seed"],
    },
    {
        "query": "গম বপনের সঠিক দূরত্ব কত?",
        "expected": ["seed"],
    },
    {
        "query": "How do I prepare a rice nursery?",
        "expected": ["seed"],
    },

    # Land preparation: 11-20
    {
        "query": "How should I prepare land for potato cultivation?",
        "expected": ["land_preparation"],
    },
    {
        "query": "How many times should the field be ploughed for wheat?",
        "expected": ["land_preparation"],
    },
    {
        "query": "What is the best way to prepare raised beds for vegetables?",
        "expected": ["land_preparation"],
    },
    {
        "query": "How do I level a rice field before planting?",
        "expected": ["land_preparation"],
    },
    {
        "query": "মরিচ চাষের জন্য জমি কীভাবে প্রস্তুত করব?",
        "expected": ["land_preparation"],
    },
    {
        "query": "ধানের জমি কতবার চাষ দিতে হবে?",
        "expected": ["land_preparation"],
    },
    {
        "query": "সবজি চাষের জন্য বেড তৈরি করব কীভাবে?",
        "expected": ["land_preparation"],
    },
    {
        "query": "কলার চারা লাগানোর গর্ত কীভাবে তৈরি করব?",
        "expected": ["land_preparation"],
    },
    {
        "query": "Is deep ploughing required for maize?",
        "expected": ["land_preparation"],
    },
    {
        "query": "How should I prepare soil before planting tomato?",
        "expected": ["land_preparation"],
    },

    # Intercultural: 21-30
    {
        "query": "How do I remove weeds manually from a rice field?",
        "expected": ["intercultural"],
    },
    {
        "query": "When should I prune tomato plants?",
        "expected": ["intercultural"],
    },
    {
        "query": "Is mulching useful for chilli cultivation?",
        "expected": ["intercultural"],
    },
    {
        "query": "How should I thin maize seedlings?",
        "expected": ["intercultural"],
    },
    {
        "query": "ধানের জমিতে হাত দিয়ে আগাছা কখন পরিষ্কার করব?",
        "expected": ["intercultural"],
    },
    {
        "query": "টমেটো গাছে খুঁটি কীভাবে দেব?",
        "expected": ["intercultural"],
    },
    {
        "query": "আলু গাছে মাটি তুলে দেওয়ার সঠিক সময় কখন?",
        "expected": ["intercultural"],
    },
    {
        "query": "বেগুন গাছ ছাঁটাই করার নিয়ম কী?",
        "expected": ["intercultural"],
    },
    {
        "query": "How often should intercultural operations be performed?",
        "expected": ["intercultural"],
    },
    {
        "query": "Should I use mulch around watermelon plants?",
        "expected": ["intercultural"],
    },

    # Irrigation: 31-40
    {
        "query": "How often should I irrigate boro rice?",
        "expected": ["irrigation"],
    },
    {
        "query": "How much water does maize need?",
        "expected": ["irrigation"],
    },
    {
        "query": "When should irrigation be stopped before harvesting wheat?",
        "expected": ["irrigation"],
    },
    {
        "query": "How can I improve drainage in a vegetable field?",
        "expected": ["irrigation"],
    },
    {
        "query": "বোরো ধানে সেচ কীভাবে দেব?",
        "expected": ["irrigation"],
    },
    {
        "query": "পেঁয়াজে কত দিন পর পর পানি দিতে হবে?",
        "expected": ["irrigation"],
    },
    {
        "query": "ফসল কাটার আগে সেচ কখন বন্ধ করব?",
        "expected": ["irrigation"],
    },
    {
        "query": "জমিতে পানি জমে গেলে কীভাবে নিষ্কাশন করব?",
        "expected": ["irrigation"],
    },
    {
        "query": "Does mustard require frequent irrigation?",
        "expected": ["irrigation"],
    },
    {
        "query": "What are the critical irrigation stages of wheat?",
        "expected": ["irrigation"],
    },

    # Harvest: 41-50
    {
        "query": "When should aman rice be harvested?",
        "expected": ["harvest"],
    },
    {
        "query": "What are the maturity signs of watermelon?",
        "expected": ["harvest"],
    },
    {
        "query": "How should wheat be harvested?",
        "expected": ["harvest"],
    },
    {
        "query": "How can harvested onions be stored?",
        "expected": ["harvest"],
    },
    {
        "query": "আমন ধান কাটার সঠিক সময় কখন?",
        "expected": ["harvest"],
    },
    {
        "query": "তরমুজ পেকেছে কীভাবে বুঝব?",
        "expected": ["harvest"],
    },
    {
        "query": "পেঁয়াজ সংগ্রহের পর কীভাবে সংরক্ষণ করব?",
        "expected": ["harvest"],
    },
    {
        "query": "এক একরে গমের ফলন কত হতে পারে?",
        "expected": ["harvest"],
    },
    {
        "query": "What is the expected yield of maize per hectare?",
        "expected": ["harvest"],
    },
    {
        "query": "How should potatoes be handled after harvesting?",
        "expected": ["harvest"],
    },

    # Fertilizer: 51-60
    {
        "query": "What fertilizer dose is recommended for boro rice?",
        "expected": ["fertilizer"],
    },
    {
        "query": "When should urea be applied to maize?",
        "expected": ["fertilizer"],
    },
    {
        "query": "Is compost beneficial for tomato cultivation?",
        "expected": ["fertilizer"],
    },
    {
        "query": "What causes nitrogen deficiency in rice?",
        "expected": ["fertilizer"],
    },
    {
        "query": "বোরো ধানে কতটুকু ইউরিয়া দিতে হবে?",
        "expected": ["fertilizer"],
    },
    {
        "query": "টমেটো চাষে গোবর সার কতটুকু দেব?",
        "expected": ["fertilizer"],
    },
    {
        "query": "ধান গাছের পাতা হলুদ হলে কোন পুষ্টির অভাব হতে পারে?",
        "expected": ["fertilizer"],
    },
    {
        "query": "গমে সার প্রয়োগের সময়সূচি কী?",
        "expected": ["fertilizer"],
    },
    {
        "query": "Which micronutrients are required for onion?",
        "expected": ["fertilizer"],
    },
    {
        "query": "Can I apply organic manure to chilli plants?",
        "expected": ["fertilizer"],
    },

    # Climate: 61-70
    {
        "query": "What temperature is suitable for wheat cultivation?",
        "expected": ["climate"],
    },
    {
        "query": "Which soil is best for tomato?",
        "expected": ["climate"],
    },
    {
        "query": "How much rainfall is required for jute?",
        "expected": ["climate"],
    },
    {
        "query": "What is the suitable soil pH for maize?",
        "expected": ["climate"],
    },
    {
        "query": "গম চাষের উপযুক্ত তাপমাত্রা কত?",
        "expected": ["climate"],
    },
    {
        "query": "টমেটো চাষের জন্য কোন মাটি ভালো?",
        "expected": ["climate"],
    },
    {
        "query": "পাট চাষে কত বৃষ্টিপাত প্রয়োজন?",
        "expected": ["climate"],
    },
    {
        "query": "ভুট্টা কোন মৌসুমে ভালো হয়?",
        "expected": ["climate"],
    },
    {
        "query": "Does cucumber require full sunlight?",
        "expected": ["climate"],
    },
    {
        "query": "What humidity level is suitable for rice?",
        "expected": ["climate"],
    },

    # Variety: 71-80
    {
        "query": "Compare different varieties of aman rice.",
        "expected": ["variety"],
    },
    {
        "query": "Which wheat variety gives the highest yield?",
        "expected": ["variety"],
    },
    {
        "query": "What are the characteristics of BRRI dhan 89?",
        "expected": ["variety"],
    },
    {
        "query": "Which tomato variety is suitable for summer?",
        "expected": ["variety"],
    },
    {
        "query": "আমন ধানের ভালো জাত কোনটি?",
        "expected": ["variety"],
    },
    {
        "query": "ব্রি ধান ৮৯ এর বৈশিষ্ট্য কী?",
        "expected": ["variety"],
    },
    {
        "query": "গমের দুটি জাতের মধ্যে তুলনা করুন",
        "expected": ["variety"],
    },
    {
        "query": "গ্রীষ্মকালীন টমেটোর জাত কী কী?",
        "expected": ["variety"],
    },
    {
        "query": "Which maize cultivar is drought tolerant?",
        "expected": ["variety"],
    },
    {
        "query": "Tell me the available varieties of mustard.",
        "expected": ["variety"],
    },

    # Pesticide: 81-88
    {
        "query": "How can I control fall armyworm in maize?",
        "expected": ["pesticide"],
    },
    {
        "query": "Which insecticide controls rice stem borer?",
        "expected": ["pesticide"],
    },
    {
        "query": "How do I treat fungal disease in tomato?",
        "expected": ["pesticide"],
    },
    {
        "query": "What causes leaf blight in rice?",
        "expected": ["pesticide"],
    },
    {
        "query": "ভুট্টার ফল আর্মিওয়ার্ম কীভাবে দমন করব?",
        "expected": ["pesticide"],
    },
    {
        "query": "ধানের মাজরা পোকার ওষুধ কী?",
        "expected": ["pesticide"],
    },
    {
        "query": "টমেটোর ছত্রাক রোগের চিকিৎসা কী?",
        "expected": ["pesticide"],
    },
    {
        "query": "বেগুনের পাতা কুঁকড়ে যাচ্ছে কেন?",
        "expected": ["pesticide"],
    },

    # Herbicide: 89-94
    {
        "query": "Which herbicide should I use in a rice field?",
        "expected": ["herbicide"],
    },
    {
        "query": "What chemical can kill weeds in maize?",
        "expected": ["herbicide"],
    },
    {
        "query": "আগাছানাশক কী?",
        "expected": ["herbicide"],
    },
    {
        "query": "ধানের জমিতে কোন আগাছানাশক ব্যবহার করব?",
        "expected": ["herbicide"],
    },
    {
        "query": "আগাছা মারার ওষুধ কখন প্রয়োগ করতে হবে?",
        "expected": ["herbicide"],
    },
    {
        "query": "How do I control weeds using chemicals?",
        "expected": ["herbicide"],
    },

    # Multiple sections: 95-97
    {
        "query": "Tell me the seed rate and fertilizer dose for wheat.",
        "expected": ["seed", "fertilizer"],
    },
    {
        "query": "How should I prepare land and irrigate maize?",
        "expected": ["land_preparation", "irrigation"],
    },
    {
        "query": "How can I remove weeds manually and also use herbicide?",
        "expected": ["intercultural", "herbicide"],
    },

    # Null cases: 98-100
    {
        "query": "বলসট",
        "expected": None,
    },
    {
        "query": "What is the capital of Bangladesh?",
        "expected": None,
    },
    {
        "query": "Hello, how are you?",
        "expected": None,
    },
        {
        "query": "আমন ধানের ভালো জাত, সার এবং সেচ সম্পর্কে বলুন",
        "expected": ["variety", "fertilizer", "irrigation"],
    },
    {
        "query": "Which herbicide controls weeds and which insecticide controls stem borer?",
        "expected": ["herbicide", "pesticide"],
    },
    {
        "query": "Compare rice varieties by yield, disease resistance and maturity time.",
        "expected": ["variety", "harvest", "pesticide"],
    },
    {
        "query": "How much rainfall is needed and when should the crop be harvested?",
        "expected": ["climate", "harvest"],
    },
    {
        "query": "Should I mulch the field, apply manure and irrigate afterward?",
        "expected": ["intercultural", "fertilizer", "irrigation"],
    },
    {
        "query": "Explain seed treatment, soil preparation and pest management.",
        "expected": ["seed", "land_preparation", "pesticide"],
    },
]


def normalize_sections(
    sections: list[SectionName] | None,
) -> list[str] | None:
    """
    Normalize the result before comparing.

    Section order is not important, so the returned list is sorted.
    """
    if not sections:
        return None

    return sorted(set(sections))


import traceback  # noqa: E402


def classify_query(query: str) -> list[SectionName] | None:
    try:
        result = classifier.invoke({"query": query})

        print("\nRAW RESULT:")
        print(repr(result))

        if result is None:
            return None

        if not result.sections:
            return None

        return result.sections

    except Exception as error:
        print("\n" + "!" * 100)
        print(f"CLASSIFICATION ERROR FOR: {query!r}")
        print(f"ERROR TYPE: {type(error).__name__}")
        print(f"ERROR: {error!r}")
        traceback.print_exc()
        print("!" * 100)

        # Do not silently convert programming/model errors into null.
        raise

def run_tests() -> list[dict]:
    results: list[dict] = []

    total = len(TEST_CASES)
    passed = 0

    for index, test_case in enumerate(TEST_CASES, start=1):
        query = test_case["query"]
        expected = test_case["expected"]

        started_at = time.perf_counter()
        actual = classify_query(query)
        elapsed = time.perf_counter() - started_at

        normalized_expected = normalize_sections(expected)
        normalized_actual = normalize_sections(actual)

        is_passed = normalized_expected == normalized_actual

        if is_passed:
            passed += 1

        result = {
            "test_number": index,
            "query": query,
            "expected": expected,
            "actual": actual,
            "passed": is_passed,
            "time_seconds": round(elapsed, 3),
        }

        results.append(result)

        status = "PASS" if is_passed else "FAIL"

        print("=" * 100)
        print(f"Test     : {index}/{total}")
        print(f"Status   : {status}")
        print(f"Query    : {query}")
        print(
            "Expected :",
            json.dumps(expected, ensure_ascii=False),
        )
        print(
            "Actual   :",
            json.dumps(actual, ensure_ascii=False),
        )
        print(f"Time     : {elapsed:.3f} seconds")

    failed = total - passed
    accuracy = (passed / total) * 100

    print("\n" + "#" * 100)
    print("FINAL TEST SUMMARY")
    print("#" * 100)
    print(f"Total tests : {total}")
    print(f"Passed      : {passed}")
    print(f"Failed      : {failed}")
    print(f"Accuracy    : {accuracy:.2f}%")

    save_results(results)

    return results


def save_results(results: list[dict]) -> None:
    with open(
        "section_test_results.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            results,
            file,
            ensure_ascii=False,
            indent=2,
        )

    failed_results = [
        result
        for result in results
        if not result["passed"]
    ]

    with open(
        "section_failed_tests.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            failed_results,
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("\nResults saved to:")
    print("- section_test_results.json")
    print("- section_failed_tests.json")


if __name__ == "__main__":
    run_tests()


RAW RESULT:
SectionResult(sections=['seed'])
Test     : 1/106
Status   : PASS
Query    : What is the seed rate for aman rice?
Expected : ["seed"]
Actual   : ["seed"]
Time     : 19.368 seconds

RAW RESULT:
SectionResult(sections=['seed'])
Test     : 2/106
Status   : PASS
Query    : How much seed is required for one acre of wheat?
Expected : ["seed"]
Actual   : ["seed"]
Time     : 6.347 seconds

RAW RESULT:
SectionResult(sections=['seed'])
Test     : 3/106
Status   : PASS
Query    : How should I treat maize seeds before sowing?
Expected : ["seed"]
Actual   : ["seed"]
Time     : 6.306 seconds

RAW RESULT:
SectionResult(sections=['seed'])
Test     : 4/106
Status   : PASS
Query    : What is the correct sowing distance for onion?
Expected : ["seed"]
Actual   : ["seed"]
Time     : 6.239 seconds

RAW RESULT:
SectionResult(sections=['seed'])
Test     : 5/106
Status   : PASS
Query    : When should I sow mustard seeds?
Expected : ["seed"]
Actual   : ["seed"]
Time     : 6.132 seconds

RAW RESULT: